In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix


In [ ]:
# right click to upload transformer.py and dataset.py to colab runtime

import sys
sys.path.append('/content')

from transformer import TransformerCls
from custom_dataset import ReviewDataset
from utils import report_metrices

In [7]:
from google.colab import drive
drive.mount("/content/drive")

proj_path = '/content/drive/MyDrive/CSE 3521/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(proj_path + "modern/custom_tokenizer.json")

In [9]:
training_set = ReviewDataset(proj_path + "data/train.csv")
val_set = ReviewDataset(proj_path + "data/val.csv")
test_set = ReviewDataset(proj_path + "data/test.csv")

training_loader = DataLoader(training_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [10]:
# sqrt-inverse class weights, normalised to mean 1
class_counts = torch.tensor(
    training_set.data["Sentiment"].value_counts().sort_index().tolist(),
    dtype=torch.float,
)

class_weights = 1.0 / torch.sqrt(class_counts)
class_weights = class_weights / class_weights.mean()

print("class counts :", class_counts.tolist())
print("class weights:", [round(w, 4) for w in class_weights.tolist()])


class counts : [4783.0, 7022.0, 20613.0]
class weights: [1.3004, 1.0732, 0.6264]


In [15]:
model_config = {
    "embedding_dim": 64,
    "nhead": 8,
    "num_layers": 2,
    "dropout": 0.3
}

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerCls(tokenizer, **model_config) 
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

In [12]:
def train_one_epoch():
    total_loss = 0
    total_samples = 0

    for i, data in enumerate(training_loader):

        inputs, labels = data
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        if i % 500 == 0:
            print(f"batch {i} loss: {total_loss / total_samples}")
    return total_loss / total_samples

In [ ]:
n_epoch = 25
patience = 3

best_f1 = -1.0
best_epoch = -1
bad_epochs = 0

for epoch in range(n_epoch):
    print(f"for epoch {epoch}")

    model.train(True)
    avg_loss = train_one_epoch()

    model.eval()
    y_val_true = []
    y_val_pred = []
    with torch.no_grad():
        for vinputs, vlabels in val_loader:
            voutputs = model(vinputs)
            y_val_pred.append(voutputs.argmax(dim=1).cpu())
            y_val_true.append(vlabels)

    y_val_true = torch.cat(y_val_true).numpy()
    y_val_pred = torch.cat(y_val_pred).numpy()
    metrices = report_metrices(y_val_true, y_val_pred)
    val_f1 = metrices["f1"]["macro"]

    scheduler.step(val_f1)

    if val_f1 > best_f1 + 1e-6:
        best_f1 = val_f1
        best_epoch = epoch
        bad_epochs = 0
        torch.save(model.state_dict(), proj_path + "modern/transformer.pth")
        print(f"  saved (best val macro-F1 {best_f1:.4f})")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"early stop: no val macro-F1 gain for {bad_epochs} epochs")
            break

print(f"best epoch {best_epoch}  val macro-F1 {best_f1:.4f}")

for epoch 0
batch 0 loss: 1.170121669769287
batch 500 loss: 1.0642568848566143
batch 1000 loss: 1.0171140372217238


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(


train loss: 1.0160  val macro-F1: 0.5810  val acc: 0.6788
  saved (best val macro-F1 0.5810)
for epoch 1
batch 0 loss: 0.8504018783569336
batch 500 loss: 0.9119786864031336
batch 1000 loss: 0.8992706386597602
train loss: 0.8993  val macro-F1: 0.6205  val acc: 0.7051
  saved (best val macro-F1 0.6205)
for epoch 2
batch 0 loss: 0.8973772525787354
batch 500 loss: 0.8629002803100083
batch 1000 loss: 0.8519377077257002
train loss: 0.8511  val macro-F1: 0.6429  val acc: 0.7332
  saved (best val macro-F1 0.6429)
for epoch 3
batch 0 loss: 0.7764363288879395
batch 500 loss: 0.829337398449104
batch 1000 loss: 0.82545193258699
train loss: 0.8254  val macro-F1: 0.6484  val acc: 0.7494
  saved (best val macro-F1 0.6484)
for epoch 4
batch 0 loss: 0.7013442516326904
batch 500 loss: 0.8033799966890179
batch 1000 loss: 0.8031610941791629
train loss: 0.8040  val macro-F1: 0.6673  val acc: 0.7601
  saved (best val macro-F1 0.6673)
for epoch 5
batch 0 loss: 0.8200361132621765
batch 500 loss: 0.78597001413

In [ ]:
model.load_state_dict(torch.load(proj_path + "modern/transformer.pth", weights_only=True))
model.eval()

y_true = []
y_pred = []
with torch.no_grad():
    for text, label in test_loader:
        output = model(text)
        y_pred.append(output.argmax(dim=1).cpu())
        y_true.append(label)

y_true = torch.cat(y_true).numpy()
y_pred = torch.cat(y_pred).numpy()
report_metrices(y_true, y_pred)

Accuracy     0.7594
F1 macro     0.6872
F1 weighted  0.7594
F1 per-class [0.678, 0.526, 0.858]
Confusion matrix (labels 0, 1, 2)
[[ 917  292  158]
 [ 262 1060  684]
 [ 159  674 5057]]
